# Get to Know a Dataset: DynaCell v1

This notebook serves as a guided tour of the [DynaCell v1](https://registry.opendata.aws/dynacell) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

**DynaCell v1** is an evaluation framework for dynamic 3D virtual staining of live cells. It contains ~677 GB of paired label-free microscopy images (3D quantitative phase and brightfield) alongside fluorescence images of tagged organelles, enabling benchmark evaluation of computational methods that predict fluorescence from label-free inputs. The data spans two cell types (A549 lung epithelial cells and hiPSCs), four organelles (plasma membrane, chromatin, ER, and mitochondria), and three viral infection states (mock, DENV, ZIKV).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of the `dynacell` S3 bucket (prefix `v1/`) we have:

| Prefix | Contents |
|---|---|
| `v1/data/` | OME-Zarr imaging archives (the core dataset) |
| `v1/metadata/` | Croissant RAI metadata and README |
| `v1/models/` | Pre-trained model checkpoint files |
| `v1/demo/` | Reviewer demo bundle (~745 MB) |
| `v1/supplementary/` | Supplementary materials |
| `v1/README.md` | Top-level dataset README |

Within `v1/data/`, the archives are split into two sub-datasets:

**`data/biohub-a549/`** (387 GB, CC-BY-4.0)
Contains 24 archives organised by organelle × condition × split:
- 4 organelles: `CAAX` (plasma membrane), `H2B` (chromatin), `SEC61B` (ER), `TOMM20` (mitochondria)
- 3 conditions: `mock` (uninfected), `DENV` (Dengue virus), `ZIKV` (Zika virus)
- 2 splits: `train/` and `test/`

Example: `data/biohub-a549/test/CAAX_mock.ozx`

**`data/aics-hipsc/`** (290 GB, Allen Institute Terms of Use)
Contains 6 archives (3 per split) of hiPSC paired imaging data.

Each archive is a single self-contained OME-Zarr plate (`.ozx` format, described in Q2).

In [ ]:
# This notebook requires the following additional libraries
# (install using pip, conda, or uv as appropriate for your environment):
#
# boto3 >= 1.38.0
# iohub >= 0.3.0
# numpy >= 2.0.0
# matplotlib >= 3.9.0

import json
import zipfile
from pprint import pprint

import boto3
import matplotlib.pyplot as plt
import numpy as np
from botocore import UNSIGNED
from botocore.config import Config
from iohub.ngff import open_ome_zarr

BUCKET = "dynacell"
PREFIX = "v1/"
BASE_URL = "https://dynacell.s3.us-west-2.amazonaws.com/v1/"

s3 = boto3.client("s3", region_name="us-west-2", config=Config(signature_version=UNSIGNED))

We start by listing the top-level prefixes and the `README.md` in the bucket.

In [ ]:
# List top-level prefixes under v1/
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")

print("Directories:")
for p in resp.get("CommonPrefixes", []):
    print(" ", p["Prefix"])

print("\nFiles:")
for o in resp.get("Contents", []):
    print(f"  {o['Key']}  ({o['Size']:,} bytes)")

In [ ]:
# List the biohub-a549 test archives
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix="v1/data/biohub-a549/test/")

print(f"{'Archive':<30} {'Size (GB)':>10}")
print("-" * 42)
for o in resp.get("Contents", []):
    name = o["Key"].split("/")[-1]
    if name:  # skip the directory entry itself
        size_gb = o["Size"] / 1e9
        print(f"{name:<30} {size_gb:>10.1f}")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

All archives use **OME-Zarr** (Open Microscopy Environment Zarr), the community standard for large bioimaging data ([OME-NGFF spec](https://ngff.openmicroscopy.org/latest/)). We use the `.ozx` file extension to signal the *OME-Zarr eXchange* format: each `.ozx` file is a **self-contained ZIP archive** wrapping a Zarr v3 store, so the entire plate ships as a single portable file.

#### Internal structure of each `.ozx` archive

Each archive is an **HCS (High-Content Screening) plate** with a single well containing 12 fields-of-view (FOVs). Every FOV stores a 5-D array with axes **(T, C, Z, Y, X)**:

| Axis | Meaning | biohub-a549 value |
|---|---|---|
| T | Timepoints (live-cell imaging) | 10 |
| C | Channels | 3 |
| Z | Z-slices (optical sections) | 48 |
| Y, X | Lateral pixels | 512 × 512 |

The three channels are:
- **Channel 0 — `Phase3D`**: 3D quantitative phase image (the label-free *input* to virtual staining models)
- **Channel 1 — `Brightfield`**: 2D transmitted-light image
- **Channel 2 — organelle target**: fluorescence image of the tagged organelle (the *output* to be predicted)

Physical voxel size: **Z = 0.174 µm**, **Y = X = 0.1494 µm** (isotropic lateral).

#### Zarr v3 with sharding

Arrays use **sharding** (a Zarr v3 feature) for efficient cloud access: the outer chunk covers one full timepoint `(1, 3, 48, 512, 512)`, and each outer chunk is sub-divided into inner shards `(1, 1, 16, 512, 512)`. Reading a single Z-slice for one channel at one timepoint fetches only the relevant shard bytes, not the full volume.

#### Normalization statistics

The root `zarr.json` stores pre-computed per-channel, per-timepoint normalization statistics (min, max, mean, std, p1, p99, …) under the `normalization` key. These percentiles are used by virtual staining models to scale inputs and outputs consistently.

#### Recommended tools

| Task | Recommended tool |
|---|---|
| HCS plate navigation | [`iohub`](https://github.com/czbiohub-sf/iohub) |
| Low-level zarr access | [`zarr`](https://zarr.readthedocs.io/) v3 |
| S3 listing / download | `boto3` or AWS CLI |
| Visualization | `napari`, `matplotlib` |

### Q: Can you show us an example of downloading and loading data from your dataset?

We will work with `CAAX_mock.ozx` — the A549 plasma membrane (CAAX) test set under mock (uninfected) conditions.

**Download the archive** using the AWS CLI (no credentials required):

```bash
aws s3 cp --no-sign-request \
    s3://dynacell/v1/data/biohub-a549/test/CAAX_mock.ozx \
    ./CAAX_mock.ozx
```

Or with `boto3`:

```python
s3.download_file("dynacell", "v1/data/biohub-a549/test/CAAX_mock.ozx", "./CAAX_mock.ozx")
```

Once downloaded, open the plate with `iohub`:

In [ ]:
# Path to the locally downloaded archive
OZX_PATH = "./CAAX_mock.ozx"

with open_ome_zarr(OZX_PATH, mode="r") as plate:
    print(f"Plate type : {type(plate).__name__}")
    print(f"\nFOVs in this plate:")
    positions = list(plate.positions())
    for name, pos in positions:
        arr = pos["0"]
        print(f"  {name}:  shape={arr.shape}  dtype={arr.dtype}")

In [ ]:
# Read plate-level metadata from the root zarr.json
with zipfile.ZipFile(OZX_PATH, "r") as zf:
    root_meta = json.loads(zf.read("zarr.json"))["attributes"]

# Plate provenance
print("=== Plate provenance ===")
for key in ("assembly_target", "assembly_condition", "assembly_split",
            "assembly_center_crop_yx", "assembly_t_cap",
            "assembly_target_yx_pixel_size_um", "assembly_contributing_plates"):
    print(f"  {key}: {root_meta[key]}")

# Channel names and dataset-level statistics for Phase3D
print("\n=== Channels with normalization stats ===")
for ch_name, stats in root_meta["normalization"].items():
    ds = stats["dataset_statistics"]
    print(f"  {ch_name}:  p1={ds['p1']:.4f}  p99={ds['p99']:.4f}  mean={ds['mean']:.4f}")

In [ ]:
# Inspect the channel labels and voxel spacing stored in FOV metadata
with zipfile.ZipFile(OZX_PATH, "r") as zf:
    fov_meta = json.loads(zf.read("0/0/fov0000/zarr.json"))["attributes"]["ome"]

channels = [ch["label"] for ch in fov_meta["omero"]["channels"]]
print("Channels:", channels)

ms = fov_meta["multiscales"][0]
axes = [(ax["name"], ax.get("unit", "-")) for ax in ms["axes"]]
scale = ms["datasets"][0]["coordinateTransformations"][0]["scale"]
print("\nAxes and voxel spacing:")
for (name, unit), s in zip(axes, scale):
    print(f"  {name}: {s} {unit}")

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

We visualise paired inputs and outputs for one FOV at the middle Z-slice:

1. **3D quantitative phase (Phase3D)** — the label-free input from which models predict fluorescence
2. **Fluorescence membrane signal (CAAX)** — the ground-truth organelle label to be predicted

We then show a Z-stack montage to convey the 3D nature of the data.

In [ ]:
def normalize_percentile(arr: np.ndarray, p1: float, p99: float) -> np.ndarray:
    """Clip and scale to [0, 1] using pre-computed percentile bounds."""
    return np.clip((arr.astype(float) - p1) / (p99 - p1 + 1e-9), 0.0, 1.0)


# Load the middle Z-slice of the first timepoint from FOV 0
with open_ome_zarr(OZX_PATH, mode="r") as plate, zipfile.ZipFile(OZX_PATH, "r") as zf:
    _, pos = next(plate.positions())
    vol = pos["0"][0]  # (C, Z, Y, X) — first timepoint

    root_meta = json.loads(zf.read("zarr.json"))["attributes"]

# Per-timepoint stats for timepoint 0
def t_stats(channel: str, stat: str) -> float:
    return root_meta["normalization"][channel]["timepoint_statistics"]["0"][stat]

z_mid = vol.shape[1] // 2  # middle Z-slice
phase = normalize_percentile(vol[0, z_mid], t_stats("Phase3D", "p1"), t_stats("Phase3D", "p99"))
fluor = normalize_percentile(vol[2, z_mid], t_stats("Membrane", "p1"), t_stats("Membrane", "p99"))

print(f"Array shape (C, Z, Y, X): {vol.shape}")
print(f"Showing Z-slice {z_mid} of {vol.shape[1]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="black")

for ax, img, title, cmap in zip(
    axes,
    [phase, fluor],
    ["Phase3D (label-free input)", "CAAX fluorescence (membrane target)"],
    ["gray", "inferno"],
):
    ax.imshow(img, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title, color="white", fontsize=13, pad=8)
    ax.axis("off")

fig.suptitle(
    "A549 cells — CAAX mock condition\n"
    f"FOV 0, timepoint 0, Z-slice {z_mid}/{vol.shape[1]}",
    color="white",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
# Z-stack montage: show every 6th Z-slice of the Phase3D channel
z_step = 6
z_indices = list(range(0, vol.shape[1], z_step))
n_cols = 6
n_rows = (len(z_indices) + n_cols - 1) // n_cols

p1_ph, p99_ph = t_stats("Phase3D", "p1"), t_stats("Phase3D", "p99")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 3 * n_rows + 0.5), facecolor="black")
axes = axes.flat

for ax, z in zip(axes, z_indices):
    ax.imshow(normalize_percentile(vol[0, z], p1_ph, p99_ph), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Z={z}", color="white", fontsize=9)
    ax.axis("off")

for ax in list(axes)[len(z_indices):]:
    ax.set_visible(False)

fig.suptitle(
    "Phase3D — Z-stack montage (FOV 0, timepoint 0, every 6th slice)",
    color="white",
    fontsize=13,
    y=1.01,
)
plt.tight_layout()
plt.show()

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Question**: Does the CAAX plasma membrane distribution look visibly different in Dengue- or Zika-infected A549 cells compared to mock-infected cells?

The dataset's design separates infection condition into distinct archives, so we can load the CAAX fluorescence channel from the same FOV index and timepoint across the three conditions and compare them side-by-side.

To run this cell you need all three archives downloaded locally:

```bash
aws s3 cp --no-sign-request s3://dynacell/v1/data/biohub-a549/test/CAAX_DENV.ozx ./CAAX_DENV.ozx
aws s3 cp --no-sign-request s3://dynacell/v1/data/biohub-a549/test/CAAX_ZIKV.ozx ./CAAX_ZIKV.ozx
```

In [ ]:
conditions = {
    "mock":   "./CAAX_mock.ozx",
    "DENV":   "./CAAX_DENV.ozx",
    "ZIKV":   "./CAAX_ZIKV.ozx",
}

FOV_IDX = 0   # same FOV across conditions
T_IDX   = 0   # same timepoint
FLUOR_CH = 2  # CAAX channel

images = {}
for cond, path in conditions.items():
    with open_ome_zarr(path, mode="r") as plate, zipfile.ZipFile(path, "r") as zf:
        positions = list(plate.positions())
        _, pos = positions[FOV_IDX]
        vol = pos["0"][T_IDX]  # (C, Z, Y, X)
        root_meta = json.loads(zf.read("zarr.json"))["attributes"]
        p1  = root_meta["normalization"]["Membrane"]["timepoint_statistics"][str(T_IDX)]["p1"]
        p99 = root_meta["normalization"]["Membrane"]["timepoint_statistics"][str(T_IDX)]["p99"]
    images[cond] = normalize_percentile(vol[FLUOR_CH, vol.shape[1] // 2], p1, p99)

fig, axes = plt.subplots(1, 3, figsize=(18, 6), facecolor="black")
for ax, (cond, img) in zip(axes, images.items()):
    ax.imshow(img, cmap="inferno", vmin=0, vmax=1)
    ax.set_title(f"CAAX — {cond}", color="white", fontsize=14, pad=8)
    ax.axis("off")

fig.suptitle(
    "CAAX membrane fluorescence across viral infection conditions\n"
    f"(FOV {FOV_IDX}, timepoint {T_IDX}, mid Z-slice)",
    color="white",
    fontsize=13,
    y=1.03,
)
plt.tight_layout()
plt.show()

The three panels reveal whether viral infection visibly perturbs plasma membrane morphology. Subtle differences in membrane intensity, distribution, or texture across conditions motivate the broader DynaCell benchmark question: can virtual staining models trained on mock-infected cells generalise to infected conditions?

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Challenge**: *Can a virtual staining model trained exclusively on mock-infected A549 cells accurately predict organelle morphology in DENV- or ZIKV-infected cells?*

This is a cross-condition generalisation problem. Viral infection can alter organelle morphology, membrane composition, and intracellular organisation in subtle ways. A model that learns to reconstruct mock-condition fluorescence from label-free phase images may fail to generalise when the underlying biology shifts due to infection.

**Recommendations**:

1. **Start with the DynaCell benchmark** — the dataset ships with evaluation configs and a pre-built pipeline (`dynacell evaluate`) that computes pixel-level similarity metrics (Spectral PCC, SSIM) and cell-level feature metrics (DINOv3 embeddings, DynaCLR features) for each (model, organelle, condition) combination. Cross-condition evaluation is supported out-of-the-box via `dynacell evaluate-grouped`.

2. **Compare in-distribution vs. out-of-distribution** — train on `*_mock` splits, evaluate on both `*_mock` (in-distribution) and `*_DENV` / `*_ZIKV` (out-of-distribution). The gap in metric scores is the generalisation penalty.

3. **Try domain adaptation** — techniques like fine-tuning on a small number of labelled infected frames, or unsupervised domain adaptation using unpaired brightfield images from infected conditions, are natural next steps.

4. **Leverage the hiPSC data** — the `aics-hipsc` sub-dataset provides a second, independent cell type. Training jointly on A549 and hiPSC data (`data/aics-hipsc/train/`) and evaluating cross-cell-type generalisation is an additional benchmark dimension.

5. **Use the model checkpoints** — pre-trained virtual staining model weights for VSCyto3D, FNet3D, UNeXt2, and CELL-Diff are provided under `v1/models/`, giving a ready-made baseline to beat.

The DynaCell codebase and benchmark configs can be found at: https://github.com/czbiohub-sf/VisCy